# TP 7 — DHCP : obtenir, vérifier et renouveler un bail

**Public cible :** étudiant, technicien, NetOps ou débutant cyber connaissant les
bases d'IPv4. **Durée : 60 minutes.** Le TP est entièrement synthétique :
aucun réseau, socket, broadcast réel, serveur, nmap, processus ou fichier.

> Mission : construire puis parser de vrais messages BOOTP/DHCP binaires en
> mémoire, suivre DORA, raisonner sur le bail, T1 et T2, et diagnostiquer sans
> toucher à une infrastructure. Toute observation sur un réseau réel exige une
> autorisation explicite.


## Contrat d'apprentissage

À la fin du TP, vous saurez :

1. raconter **DORA** : **Discover → Offer → Request → ACK** ;
2. reconnaître le magic cookie DHCP et les options TLV dans de vrais octets ;
3. expliquer broadcast, unicast et le rôle d'un **relais DHCP** avec giaddr ;
4. vérifier masque, routeur, DNS, durée du **bail**, **T1** et **T2** ;
5. distinguer absence d'Offer, NAK, **pool épuisé** et option incohérente ;
6. formuler un diagnostic par étape et demander la prochaine preuve utile.

Avant chaque révélation, prédisez : quel message doit suivre, qui l'envoie et
quelle option prouverait votre hypothèse ?


In [ ]:
import base64
import hashlib
import ipaddress
import struct

SEED = 0xD0A4
MAGIC_COOKIE = bytes.fromhex("63825363")
DHCP_TYPES = {
    1: "Discover", 2: "Offer", 3: "Request", 4: "Decline",
    5: "ACK", 6: "NAK", 7: "Release", 8: "Inform",
}
OPTION_NAMES = {
    1: "subnet_mask", 3: "router", 6: "dns", 50: "requested_ip",
    51: "lease_seconds", 53: "message_type", 54: "server_id",
    58: "t1_seconds", 59: "t2_seconds",
}
BOOTREQUEST, BOOTREPLY = 1, 2
FIXED_FORMAT = "!BBBBIHH4s4s4s4s16s64s128s"
FIXED_LEN = struct.calcsize(FIXED_FORMAT)
assert FIXED_LEN == 236

def ip_bytes(address):
    return ipaddress.IPv4Address(address).packed

def ip_text(raw):
    return str(ipaddress.IPv4Address(raw))

def mac_bytes(address):
    raw = bytes.fromhex(address.replace(":", ""))
    assert len(raw) == 6
    return raw

def option(code, payload):
    payload = bytes(payload)
    assert 0 < code < 255 and len(payload) <= 255
    return bytes((code, len(payload))) + payload

def u32(value):
    return struct.pack("!I", value)

def build_dhcp(message_type, *, op, xid, mac, yiaddr="0.0.0.0",
               giaddr="0.0.0.0", flags=0x8000, hops=0, options=()):
    chaddr = mac_bytes(mac) + bytes(10)
    fixed = struct.pack(
        FIXED_FORMAT,
        op, 1, 6, hops, xid, 0, flags,
        ip_bytes("0.0.0.0"), ip_bytes(yiaddr),
        ip_bytes("0.0.0.0"), ip_bytes(giaddr),
        chaddr, bytes(64), bytes(128),
    )
    encoded = option(53, bytes((message_type,)))
    encoded += b"".join(option(code, payload) for code, payload in options)
    packet = fixed + MAGIC_COOKIE + encoded + bytes((255,))
    assert packet[236:240] == MAGIC_COOKIE
    return packet

def parse_options(raw):
    parsed = {}
    cursor = 0
    while cursor < len(raw):
        code = raw[cursor]
        cursor += 1
        if code == 255:
            break
        if code == 0:
            continue
        assert cursor < len(raw), "option DHCP sans longueur"
        size = raw[cursor]
        cursor += 1
        assert cursor + size <= len(raw), "option DHCP tronquée"
        parsed.setdefault(code, []).append(raw[cursor:cursor + size])
        cursor += size
    return parsed

def decode_option(code, payload):
    if code == 53:
        return DHCP_TYPES[payload[0]]
    if code in (1, 50, 54):
        return ip_text(payload)
    if code in (3, 6):
        assert len(payload) % 4 == 0
        return [ip_text(payload[i:i + 4]) for i in range(0, len(payload), 4)]
    if code in (51, 58, 59):
        return struct.unpack("!I", payload)[0]
    return payload.hex()

def parse_dhcp(packet):
    assert len(packet) >= 241, "message trop court"
    fields = struct.unpack(FIXED_FORMAT, packet[:FIXED_LEN])
    assert packet[FIXED_LEN:FIXED_LEN + 4] == MAGIC_COOKIE, "magic cookie absent"
    raw_options = parse_options(packet[FIXED_LEN + 4:])
    assert 53 in raw_options, "type DHCP absent"
    decoded = {
        OPTION_NAMES.get(code, f"option_{code}"): decode_option(code, values[0])
        for code, values in raw_options.items()
    }
    return {
        "op": fields[0],
        "hops": fields[3],
        "xid": fields[4],
        "flags": fields[6],
        "ciaddr": ip_text(fields[7]),
        "yiaddr": ip_text(fields[8]),
        "siaddr": ip_text(fields[9]),
        "giaddr": ip_text(fields[10]),
        "chaddr": ":".join(f"{byte:02x}" for byte in fields[11][:fields[2]]),
        "options": decoded,
        "length": len(packet),
    }

assert parse_options(option(53, b"\x01") + b"\xff")[53][0] == b"\x01"
print(f"Codec prêt : en-tête BOOTP={FIXED_LEN} octets, cookie={MAGIC_COOKIE.hex()}")


## 1. Construire DORA, octet par octet

Le client ne possède encore ni adresse ni connaissance certaine du serveur.

| Étape | Sens | Transport logique | Preuve décisive |
|---|---|---|---|
| Discover | client → serveurs | broadcast | option 53 = 1 |
| Offer | serveur → client | souvent broadcast au départ | yiaddr + serveur + bail |
| Request | client → serveurs | broadcast pendant la sélection | adresse demandée + serveur choisi |
| ACK | serveur → client | broadcast ou unicast selon le contexte | configuration et temporisateurs |

Le broadcast permet aux serveurs non retenus de voir le choix. Après
configuration, le renouvellement T1 vise normalement le serveur en unicast ;
le rebinding T2 revient en broadcast si ce serveur ne répond pas.


In [ ]:
CLIENT_MAC = "02:00:00:00:00:42"
CLIENT_IP = "192.0.2.42"
SERVER_IP = "192.0.2.1"
ROUTER_IP = "192.0.2.254"
DNS_IPS = ("192.0.2.53", "198.51.100.53")
MASK = "255.255.255.0"
LEASE, T1, T2 = 3600, 1800, 3150
XID = (SEED << 16) | 0x0042

lease_options = (
    (54, ip_bytes(SERVER_IP)),
    (1, ip_bytes(MASK)),
    (3, ip_bytes(ROUTER_IP)),
    (6, b"".join(ip_bytes(address) for address in DNS_IPS)),
    (51, u32(LEASE)), (58, u32(T1)), (59, u32(T2)),
)

discover = build_dhcp(
    1, op=BOOTREQUEST, xid=XID, mac=CLIENT_MAC,
)
offer = build_dhcp(
    2, op=BOOTREPLY, xid=XID, mac=CLIENT_MAC, yiaddr=CLIENT_IP,
    options=lease_options,
)
request = build_dhcp(
    3, op=BOOTREQUEST, xid=XID, mac=CLIENT_MAC,
    options=((50, ip_bytes(CLIENT_IP)), (54, ip_bytes(SERVER_IP))),
)
ack = build_dhcp(
    5, op=BOOTREPLY, xid=XID, mac=CLIENT_MAC, yiaddr=CLIENT_IP,
    options=lease_options,
)

DORA_PACKETS = (discover, offer, request, ack)
DORA = tuple(parse_dhcp(packet) for packet in DORA_PACKETS)

assert [row["options"]["message_type"] for row in DORA] == [
    "Discover", "Offer", "Request", "ACK"
]
assert len({row["xid"] for row in DORA}) == 1
assert DORA[1]["yiaddr"] == CLIENT_IP
assert DORA[2]["options"]["requested_ip"] == CLIENT_IP
assert DORA[3]["options"]["dns"] == list(DNS_IPS)
assert 0 < T1 < T2 < LEASE

for index, row in enumerate(DORA, 1):
    direction = "client → serveur(s)" if row["op"] == BOOTREQUEST else "serveur → client"
    print(
        f"{index}. {row['options']['message_type']:<8} {direction:<20} "
        f"{row['length']} octets xid=0x{row['xid']:08x}"
    )


## 2. Lire les champs, pas seulement les noms

Dans le scénario direct, giaddr vaut 0.0.0.0. Un **relais DHCP** reçoit le
broadcast local, place son adresse d'interface dans giaddr, incrémente hops puis
transmet vers un serveur distant. Le serveur utilise giaddr pour choisir le
pool du bon sous-réseau et renvoie sa réponse via le relais.

Dans l'ACK, vérifiez ensemble :

- masque et routeur pour l'attachement local ;
- DNS pour la résolution de noms ;
- bail pour la durée de validité ;
- T1 pour le renouvellement et T2 pour le rebinding.

Une option reçue n'est pas automatiquement cohérente : elle doit respecter le
contexte d'adressage et la relation 0 < T1 < T2 < bail.


In [ ]:
RELAY_IP = "10.10.0.1"
relayed_discover = build_dhcp(
    1, op=BOOTREQUEST, xid=XID + 1, mac=CLIENT_MAC,
    giaddr=RELAY_IP, flags=0, hops=1,
)
relay_view = parse_dhcp(relayed_discover)

def same_subnet(address, peer, mask):
    network = ipaddress.IPv4Network(f"{address}/{mask}", strict=False)
    return ipaddress.IPv4Address(peer) in network

def validate_ack(view):
    options = view["options"]
    errors = []
    lease = options.get("lease_seconds", 0)
    t1 = options.get("t1_seconds", 0)
    t2 = options.get("t2_seconds", 0)
    if not (0 < t1 < t2 < lease):
        errors.append("temporisateurs attendus : 0 < T1 < T2 < bail")
    routers = options.get("router", [])
    if not routers or not same_subnet(view["yiaddr"], routers[0], options["subnet_mask"]):
        errors.append("routeur hors du sous-réseau annoncé")
    if not options.get("dns"):
        errors.append("aucun DNS annoncé")
    return errors

assert relay_view["giaddr"] == RELAY_IP
assert relay_view["hops"] == 1
assert validate_ack(DORA[3]) == []

print(
    "Relais synthétique : "
    f"giaddr={relay_view['giaddr']}, hops={relay_view['hops']}, "
    f"xid=0x{relay_view['xid']:08x}"
)
print("ACK cohérent :", ", ".join(
    f"{name}={value}" for name, value in DORA[3]["options"].items()
))


## 3. Du bail au premier paquet : sous-réseau ou passerelle ?

L'ACK donne `192.0.2.42`, le masque `255.255.255.0` — soit `/24` — et la **passerelle par défaut** `192.0.2.254`. L'ET bit à bit entre adresse et masque donne le réseau `192.0.2.0`; son broadcast IPv4 ordinaire est `192.0.2.255`.

La machine consulte ensuite sa table de routage selon le **préfixe le plus long** :

- cible dans la route connectée `/24` : le prochain saut est la cible ; ARP résout sa MAC ;
- cible distante : la route par défaut gagne ; ARP résout la MAC de la gateway ;
- dans les deux cas, l'IP de destination reste celle de la cible finale. Seule la MAC de la trame désigne le prochain saut.

En IPv6, une route `::/0` et Neighbor Discovery jouent les rôles correspondants ; il n'existe pas de broadcast IPv6. Gateway, DNS, NAT et pare-feu sont des fonctions distinctes même si une box les cumule.


In [ ]:
client_interface = ipaddress.IPv4Interface(f"{CLIENT_IP}/{MASK}")
connected_network = client_interface.network
route_table = (
    {"prefix": connected_network, "via": None, "name": "route connectée"},
    {"prefix": ipaddress.IPv4Network("0.0.0.0/0"), "via": ipaddress.IPv4Address(ROUTER_IP), "name": "route par défaut"},
)

def forwarding_decision(destination):
    final_ip = ipaddress.IPv4Address(destination)
    compatible = [route for route in route_table if final_ip in route["prefix"]]
    route = max(compatible, key=lambda item: item["prefix"].prefixlen)
    next_hop = final_ip if route["via"] is None else route["via"]
    return {
        "destination_ip": str(final_ip),
        "winning_route": str(route["prefix"]),
        "decision": route["name"],
        "next_hop": str(next_hop),
        "arp_question": f"qui a {next_hop} ?",
        "ethernet_recipient": "destination" if route["via"] is None else "gateway",
    }

forwarding_decisions = tuple(map(forwarding_decision, ("192.0.2.99", "198.51.100.10")))
assert str(connected_network.network_address) == "192.0.2.0"
assert str(connected_network.broadcast_address) == "192.0.2.255"
assert forwarding_decisions[0]["next_hop"] == "192.0.2.99"
assert forwarding_decisions[1]["destination_ip"] == "198.51.100.10"
assert forwarding_decisions[1]["next_hop"] == ROUTER_IP

for decision in forwarding_decisions:
    print(
        f"{decision['destination_ip']:<15} → {decision['winning_route']:<16} "
        f"→ ARP {decision['next_hop']:<15} → MAC {decision['ethernet_recipient']}"
    )


## 4. Lire l'horloge du bail

Pour un ACK reçu à t=0 dans ce TP :

- avant T1 : l'adresse reste valide ;
- à T1=1800 s : renouvellement unicast vers le serveur connu ;
- à T2=3150 s : sans réponse, rebinding en broadcast vers tout serveur ;
- à 3600 s : expiration ; le client ne doit plus utiliser l'adresse.

Cette horloge enseigne les états, pas les détails complets d'un système
d'exploitation. Avant de révéler le schéma, placez T1 et T2 sur une ligne allant
de 0 au bail.


In [ ]:
class RichHTML:
    def __init__(self, markup):
        self.markup = markup

    def _repr_html_(self):
        return self.markup

    def __repr__(self):
        return "RichHTML(pedagogical-output)"

def dhcp_svg():
    steps = [
        ("Discover", 150, 330, "#38bdf8"),
        ("Offer", 330, 150, "#a78bfa"),
        ("Request", 150, 330, "#f59e0b"),
        ("ACK", 330, 150, "#34d399"),
    ]
    lines = [
        '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1120 720" role="img" '
        'aria-labelledby="dhcp-svg-title dhcp-svg-desc">',
        '<title id="dhcp-svg-title">DORA et horloge du bail DHCP</title>',
        '<desc id="dhcp-svg-desc">Waterfall des quatre messages et jalons T1, T2 et expiration.</desc>',
        '<rect width="1120" height="720" rx="28" fill="#07111f"/>',
        '<text x="55" y="58" fill="#f8fafc" font-size="30" font-family="sans-serif" font-weight="700">'
        'DHCP — des octets à la décision</text>',
        '<text x="150" y="110" fill="#bae6fd" font-size="20" font-family="sans-serif">Client</text>',
        '<text x="875" y="110" fill="#ddd6fe" font-size="20" font-family="sans-serif">Serveur</text>',
        '<line x1="180" y1="130" x2="180" y2="450" stroke="#475569" stroke-width="3"/>',
        '<line x1="930" y1="130" x2="930" y2="450" stroke="#475569" stroke-width="3"/>',
    ]
    for index, (label, start_y, end_y, color) in enumerate(steps):
        y = 175 + index * 68
        left_to_right = index in (0, 2)
        x1, x2 = (180, 930) if left_to_right else (930, 180)
        tip = x2 - 14 if left_to_right else x2 + 14
        lines.extend([
            f'<line x1="{x1}" y1="{y}" x2="{x2}" y2="{y}" stroke="{color}" stroke-width="5"/>',
            f'<polygon points="{x2},{y} {tip},{y - 9} {tip},{y + 9}" fill="{color}"/>',
            f'<text x="505" y="{y - 12}" fill="{color}" font-size="19" '
            f'font-family="sans-serif" font-weight="700">{label}</text>',
        ])
    x0, width, y = 100, 920, 560
    lines.extend([
        f'<line x1="{x0}" y1="{y}" x2="{x0 + width}" y2="{y}" stroke="#94a3b8" stroke-width="7"/>',
        '<text x="55" y="510" fill="#f8fafc" font-size="23" font-family="sans-serif" '
        'font-weight="700">Horloge du bail</text>',
    ])
    for seconds, label, color in (
        (0, "ACK", "#34d399"), (T1, "T1 unicast", "#38bdf8"),
        (T2, "T2 broadcast", "#f59e0b"), (LEASE, "expiration", "#ef4444"),
    ):
        x = x0 + width * seconds / LEASE
        lines.extend([
            f'<line x1="{x:.1f}" y1="{y - 24}" x2="{x:.1f}" y2="{y + 24}" '
            f'stroke="{color}" stroke-width="5"/>',
            f'<text x="{x:.1f}" y="{y + 58}" text-anchor="middle" fill="{color}" '
            f'font-size="17" font-family="sans-serif">{label}</text>',
            f'<text x="{x:.1f}" y="{y + 82}" text-anchor="middle" fill="#cbd5e1" '
            f'font-size="15" font-family="sans-serif">{seconds} s</text>',
        ])
    lines.extend([
        '<rect x="55" y="664" width="1010" height="35" rx="10" fill="#10243b"/>',
        '<text x="560" y="688" text-anchor="middle" fill="#e2e8f0" font-size="16" '
        'font-family="sans-serif">T1 &lt; T2 &lt; bail : renouveler, rebinder, puis cesser l’usage</text>',
        '</svg>',
    ])
    return "".join(lines)

assert "<svg" in dhcp_svg() and "T1 unicast" in dhcp_svg()
RichHTML(dhcp_svg())


## 5. Diagnostiquer l'étape qui casse

Un symptôme « pas d'adresse » ne suffit pas. Localisez d'abord la dernière
preuve valide, puis demandez une preuve indépendante sur un banc autorisé.

| Incident synthétique | Dernière preuve | Diagnostic d'étape | Prochaine preuve |
|---|---|---|---|
| absence d'Offer | Discover émis | panne avant Offer | trace côté relais/serveur |
| NAK | Request émis, NAK reçu | refus explicite du serveur | adresse demandée et server-id |
| pool épuisé | Discover reçu côté serveur | aucune adresse libre | compteur du pool et baux actifs |
| option incohérente | ACK reçu | configuration reçue mais invalide | masque, routeur, DNS, T1/T2/bail |

Ne contournez jamais une politique ou un pool. L'observation d'un service DHCP
réel requiert autorisation, périmètre et journalisation.


In [ ]:
nak = build_dhcp(
    6, op=BOOTREPLY, xid=XID, mac=CLIENT_MAC,
    options=((54, ip_bytes(SERVER_IP)),),
)
bad_ack = build_dhcp(
    5, op=BOOTREPLY, xid=XID + 2, mac=CLIENT_MAC, yiaddr=CLIENT_IP,
    options=(
        (54, ip_bytes(SERVER_IP)),
        (1, ip_bytes(MASK)),
        (3, ip_bytes("203.0.113.254")),
        (6, ip_bytes(DNS_IPS[0])),
        (51, u32(2400)), (58, u32(1800)), (59, u32(1200)),
    ),
)

INCIDENTS = {
    "absence_offer": {
        "trace": (discover, discover),
        "stage": "Discover → Offer",
        "observation": "deux Discover, aucun Offer observé",
        "diagnostic": "la séquence s'arrête avant Offer",
        "next_proof": "corréler xid/giaddr avec la trace du relais puis le journal serveur",
    },
    "nak": {
        "trace": (discover, offer, request, nak),
        "stage": "Request → ACK",
        "observation": "un NAK répond au Request",
        "diagnostic": "refus explicite de la configuration demandée",
        "next_proof": "comparer requested_ip, server_id et état du bail",
    },
    "pool_epuise": {
        "trace": (discover,),
        "stage": "Discover → Offer",
        "observation": "le serveur confirme zéro adresse libre dans le scénario",
        "diagnostic": "pool épuisé, donc aucune Offer possible",
        "next_proof": "compter baux actifs, expirés et adresses réservées côté serveur",
    },
    "option_incoherente": {
        "trace": (discover, offer, request, bad_ack),
        "stage": "validation de l'ACK",
        "observation": "T2 précède T1 et le routeur sort du sous-réseau",
        "diagnostic": "ACK syntaxiquement lisible mais configuration incohérente",
        "next_proof": "comparer options du scope, politique serveur et ACK brut",
    },
}

def trace_types(packets):
    return [parse_dhcp(packet)["options"]["message_type"] for packet in packets]

def diagnose_incident(name, incident):
    types = trace_types(incident["trace"])
    if name == "absence_offer":
        assert types == ["Discover", "Discover"]
    elif name == "nak":
        assert types[-1] == "NAK"
    elif name == "pool_epuise":
        assert types == ["Discover"]
    elif name == "option_incoherente":
        errors = validate_ack(parse_dhcp(incident["trace"][-1]))
        assert len(errors) == 2
    return {
        "incident": name,
        "messages": types,
        "stage": incident["stage"],
        "diagnostic": incident["diagnostic"],
        "next_proof": incident["next_proof"],
    }

INCIDENT_RESULTS = [
    diagnose_incident(name, incident) for name, incident in INCIDENTS.items()
]
assert len(INCIDENT_RESULTS) == 4
for result in INCIDENT_RESULTS:
    print(
        f"{result['incident']:<20} | {result['stage']:<20} | "
        f"{result['diagnostic']}\n  prochaine preuve : {result['next_proof']}"
    )


## 6. Cas de transfert — la valeur X

Un poste situé derrière un relais présente cette trace inédite :

1. Discover avec giaddr=10.10.0.1 ;
2. Offer pour 10.10.0.77 ;
3. Request de 10.10.0.77 ;
4. NAK du serveur choisi.

Sans réutiliser aveuglément les adresses du premier scénario :

- identifiez la transition qui échoue ;
- expliquez pourquoi le relais n'est pas prouvé défaillant ;
- proposez la prochaine preuve, sans lancer de scan ni de serveur ;
- dites ce que le client doit refaire après le NAK.

Le bloc suivant vérifie le raisonnement sur de nouveaux octets.


In [ ]:
TRANSFER_XID = XID + 99
TRANSFER_MAC = "02:00:00:00:10:77"
TRANSFER_CLIENT = "10.10.0.77"
TRANSFER_SERVER = "198.51.100.10"

transfer_packets = (
    build_dhcp(
        1, op=BOOTREQUEST, xid=TRANSFER_XID, mac=TRANSFER_MAC,
        giaddr=RELAY_IP, flags=0, hops=1,
    ),
    build_dhcp(
        2, op=BOOTREPLY, xid=TRANSFER_XID, mac=TRANSFER_MAC,
        yiaddr=TRANSFER_CLIENT, giaddr=RELAY_IP, flags=0, hops=1,
        options=((54, ip_bytes(TRANSFER_SERVER)), (51, u32(7200))),
    ),
    build_dhcp(
        3, op=BOOTREQUEST, xid=TRANSFER_XID, mac=TRANSFER_MAC,
        giaddr=RELAY_IP, flags=0, hops=1,
        options=((50, ip_bytes(TRANSFER_CLIENT)), (54, ip_bytes(TRANSFER_SERVER))),
    ),
    build_dhcp(
        6, op=BOOTREPLY, xid=TRANSFER_XID, mac=TRANSFER_MAC,
        giaddr=RELAY_IP, flags=0, hops=1,
        options=((54, ip_bytes(TRANSFER_SERVER)),),
    ),
)
transfer_views = tuple(parse_dhcp(packet) for packet in transfer_packets)
transfer_types = [view["options"]["message_type"] for view in transfer_views]

assert transfer_types == ["Discover", "Offer", "Request", "NAK"]
assert all(view["giaddr"] == RELAY_IP for view in transfer_views)
transfer_diagnosis = {
    "failed_transition": "Request → ACK",
    "reasoning": "Offer et NAK ont traversé le relais ; un silence du relais n'est pas démontré",
    "next_proof": "vérifier requested_ip et server_id dans le journal autorisé du serveur",
    "client_action": "abandonner l'adresse et recommencer DORA",
}
print("Cas X :", transfer_diagnosis)

def packet_record(label, packet):
    view = parse_dhcp(packet)
    return (
        f"{label}|type={view['options']['message_type']}|xid=0x{view['xid']:08x}|"
        f"giaddr={view['giaddr']}|length={len(packet)}|"
        f"sha256={hashlib.sha256(packet).hexdigest()}|hex={packet.hex()}"
    )

dhcp_trace = "\n".join(
    packet_record(f"DORA-{index}", packet)
    for index, packet in enumerate(DORA_PACKETS, 1)
)
dhcp_trace += "\n" + "\n".join(
    packet_record(f"TRANSFER-{index}", packet)
    for index, packet in enumerate(transfer_packets, 1)
)

incident_report = """# Rapport d'incident DHCP synthétique

## Méthode
Localiser la dernière étape DORA prouvée, parser les options, vérifier le bail,
T1 et T2, puis demander une preuve indépendante dans le périmètre autorisé.

## Résultats
""" + "\n".join(
    f"- {row['incident']}: étape {row['stage']}; {row['diagnostic']}. "
    f"Prochaine preuve : {row['next_proof']}."
    for row in INCIDENT_RESULTS
) + f"""

## Cas de transfert
- Transition : {transfer_diagnosis['failed_transition']}
- Raisonnement : {transfer_diagnosis['reasoning']}
- Prochaine preuve : {transfer_diagnosis['next_proof']}
- Action client : {transfer_diagnosis['client_action']}

## Limites
Octets et incidents synthétiques ; aucune capture ni preuve de production.
"""

def data_link(label, filename, text, mime):
    payload = base64.b64encode(text.encode("utf-8")).decode("ascii")
    return (
        f'<a download="{filename}" href="data:{mime};base64,{payload}" '
        f'style="display:inline-block;margin:8px;padding:10px 15px;border-radius:9px;'
        f'background:#0f766e;color:white;text-decoration:none">{label}</a>'
    )

assert "DORA-1|type=Discover" in dhcp_trace
assert "Cas de transfert" in incident_report
assert hashlib.sha256(dhcp_trace.encode()).hexdigest() == hashlib.sha256(
    dhcp_trace.encode()
).hexdigest()

exports_html = (
    '<div style="font-family:sans-serif;padding:14px;border:1px solid #94a3b8;'
    'border-radius:12px"><strong>Artefacts pédagogiques en mémoire</strong><br>'
    + data_link("Télécharger dhcp_trace", "dhcp_trace.txt", dhcp_trace, "text/plain")
    + data_link(
        "Télécharger incident_report",
        "incident_report.md",
        incident_report,
        "text/markdown",
    )
    + "</div>"
)
RichHTML(exports_html)


## 7. Auto-évaluation, autorisation et limites

Barème conseillé sur 10 : structure binaire et cookie (2), DORA et sens des
échanges (2), bail/T1/T2 (2), diagnostic par étape (2), cas de transfert et
prochaine preuve (2).

### Limites

- messages BOOTP/DHCP réellement encodés et parsés, mais uniquement en mémoire ;
- aucun UDP, broadcast, unicast, serveur ou relais DHCP réellement exécuté ;
- pas de temporisation réelle, retransmission complète, état persistant,
  DHCPv6, option 82, authentification ni haute disponibilité ;
- incidents et compteurs entièrement synthétiques ; ni capture terrain, ni
  preuve de production, ni conclusion cyber opérationnelle ;
- une adresse Offer n'est pas encore un bail utilisable ; seul l'ACK termine
  ici l'allocation, et un NAK impose l'abandon ;
- l'export data URI est un artefact pédagogique local, pas un fichier écrit par
  le notebook.

Sur un laboratoire réel, obtenir l'**autorisation**, borner le périmètre,
documenter versions et horodatages, puis corréler capture, relais, serveur et
client. Ne lancez jamais de serveur DHCP sauvage.
